- CNB, 로지스틱 회귀, 서포트 벡터 머신, 랜덤 포레스트
- 딥러닝 모델과 비교

In [26]:
import tensorflow
import tensorflow.keras.datasets.reuters as reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib
import seaborn
import numpy
import pandas
import sklearn

print(tensorflow.__version__)
print(matplotlib.__version__)
print(seaborn.__version__)
print(numpy.__version__)
print(pandas.__version__)
print(sklearn.__version__)

2.15.1
3.10.3
0.13.2
1.26.4
2.3.1
1.7.1


- 데이터 다운로드

In [27]:
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=5000, test_split=0.2)

print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)

(8982,) (2246,)
(8982,) (2246,)


- word_index 다운로드

In [28]:
# word_index = reuters.get_word_index(path="reuters_word_index.json")
!wget https://storage.googleapis.com/tensorflow/tf-keras-datasets/reuters_word_index.json

import json

# 1. 파일 열기
with open('reuters_word_index.json', 'r', encoding='utf-8') as f:
    # 2. JSON 로드
    word_index = json.load(f)

## 0, 1, 2는 <pad>, <sos>, <unk>토큰을 나타내므로 index에 +3을 해줘야함
index_to_word = { index+3 : word for word, index in word_index.items() }

# index_to_word에 숫자 0은 <pad>, 숫자 1은 <sos>, 숫자 2는 <unk>
for index, token in enumerate(("<pad>", "<sos>", "<unk>")):
  index_to_word[index]=token

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
--2025-09-01 15:26:42--  https://storage.googleapis.com/tensorflow/tf-keras-datasets/reuters_word_index.json
Resolving storage.googleapis.com (storage.googleapis.com)... 34.128.10.91, 34.128.9.187, 34.128.9.251, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.128.10.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 550378 (537K) [application/json]
Saving to: 'reuters_word_index.json.4'

reuters_word_index. 100%[===================>] 537.48K  --.-KB/s    in 0.05s   

2025-09-01 15:26:42 (10.4 MB/s) - 'reuters_word_index.json.4' saved [550378/550378]



- 데이터 index -> word

In [29]:
def decode_data(x_df, index_to_word):
    decoded = []
    for i in range(len(x_df)):
        t = ' '.join([index_to_word[index] for index in x_df[i]])
        decoded.append(t)

    return decoded


x_train = decode_data(x_train, index_to_word)
x_test = decode_data(x_test, index_to_word)

print(len(x_train))
print(len(x_test))

8982
2246


In [30]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

def get_tfidf(x_train, x_test):
    dtmvector = CountVectorizer()
    x_train_dtm = dtmvector.fit_transform(x_train)
    x_test_dtm = dtmvector.transform(x_test)

    tfidf_transformer = TfidfTransformer()
    tfidfv_train = tfidf_transformer.fit_transform(x_train_dtm)
    tfidfv_test = tfidf_transformer.transform(x_test_dtm) #DTM을 TF-IDF 행렬로 변환

    return tfidfv_train, tfidfv_test


tfidfv_train, tfidfv_test = get_tfidf(x_train, x_test)
print(tfidfv_train.shape, tfidfv_test.shape)

(8982, 4867) (2246, 4867)


- Vocab size별로 실험

In [66]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from lightgbm import LGBMClassifier

models = {
    "LogReg" : LogisticRegression(max_iter=200, n_jobs=-1),
    "SGD_hinge" : SGDClassifier(loss="hinge", max_iter=1000, tol=1e-3, random_state=42),
    "MultinNB"  : MultinomialNB(alpha=1.0),
    "CompNB" : ComplementNB(),
    "DecTree"   : DecisionTreeClassifier(random_state=42, max_depth=None),
    "RandFor": RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42),
    "LGBM" : LGBMClassifier(iteration=300, random_state=42, verbose=-1, n_jobs=-1),
    "LinSVC" : LinearSVC(max_iter=5000, random_state=42)
}

for model_name, model_template in models.items():
    print('\n')
    vocal_size_list = [3000, 5000, None]
    for vocal_size in vocal_size_list:
        (x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=vocal_size, test_split=0.2)

        x_train = decode_data(x_train, index_to_word)
        x_test = decode_data(x_test, index_to_word)

        tfidfv_train, tfidfv_test = get_tfidf(x_train, x_test)

        if model_name == "LGBM":
            tfidfv_train = tfidfv_train.toarray()
            tfidfv_test  = tfidfv_test.toarray()

        # 모델 복사(재사용 방지)
        clf = model_template.__class__(**model_template.get_params())
        clf.fit(tfidfv_train, y_train)

        predicted = clf.predict(tfidfv_test)
        acc = accuracy_score(y_test, predicted)
        f1 = f1_score(y_test, predicted, average="macro")
        print(f"[Vocab size {vocal_size} | {model_name}] 정확도: {acc:.4f}, F1 score: {f1:.4f}")



[Vocab size 3000 | LogReg] 정확도: 0.7988, F1 score: 0.4831
[Vocab size 5000 | LogReg] 정확도: 0.7979, F1 score: 0.4814
[Vocab size None | LogReg] 정확도: 0.7916, F1 score: 0.4514


[Vocab size 3000 | SGD_hinge] 정확도: 0.8419, F1 score: 0.6826
[Vocab size 5000 | SGD_hinge] 정확도: 0.8411, F1 score: 0.6786
[Vocab size None | SGD_hinge] 정확도: 0.8428, F1 score: 0.6826


[Vocab size 3000 | MultinNB] 정확도: 0.6874, F1 score: 0.1435
[Vocab size 5000 | MultinNB] 정확도: 0.6732, F1 score: 0.1102
[Vocab size None | MultinNB] 정확도: 0.5997, F1 score: 0.0677


[Vocab size 3000 | CompNB] 정확도: 0.7645, F1 score: 0.4426
[Vocab size 5000 | CompNB] 정확도: 0.7707, F1 score: 0.4820
[Vocab size None | CompNB] 정확도: 0.7649, F1 score: 0.4640


[Vocab size 3000 | DecTree] 정확도: 0.7061, F1 score: 0.4641
[Vocab size 5000 | DecTree] 정확도: 0.6995, F1 score: 0.4480
[Vocab size None | DecTree] 정확도: 0.6963, F1 score: 0.4503


[Vocab size 3000 | RandFor] 정확도: 0.7760, F1 score: 0.4812
[Vocab size 5000 | RandFor] 정확도: 0.7663, F1 score: 0.4587

/home/eunhak_linux/miniconda3/envs/aiffel/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[Vocab size 3000 | LGBM] 정확도: 0.4524, F1 score: 0.0329


/home/eunhak_linux/miniconda3/envs/aiffel/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[Vocab size 5000 | LGBM] 정확도: 0.3246, F1 score: 0.0109


/home/eunhak_linux/miniconda3/envs/aiffel/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[Vocab size None | LGBM] 정확도: 0.1923, F1 score: 0.0142


[Vocab size 3000 | LinSVC] 정확도: 0.8290, F1 score: 0.6678
[Vocab size 5000 | LinSVC] 정확도: 0.8290, F1 score: 0.6814
[Vocab size None | LinSVC] 정확도: 0.8295, F1 score: 0.6887


| 모델             | Vocab 3 k           | Vocab 5 k           | Vocab All           | **최고 F1**        | **최고 Acc**       |
| -------------- | ------------------- | ------------------- | ------------------- | ---------------- | ---------------- |
| **LogReg**     | 0.7988 / 0.4831     | 0.7979 / 0.4814     | 0.7916 / 0.4514     | 0.4831 (3 k)     | 0.7988 (3 k)     |
| **SGD-hinge**  | 0.8419 / 0.6826     | 0.8411 / 0.6786     | **0.8428 / 0.6826** | 0.6826 (3 k≒All) | **0.8428 (All)** |
| **MultinNB**   | 0.6874 / 0.1435     | 0.6732 / 0.1102     | 0.5997 / 0.0677     | 0.1435 (3 k)     | 0.6874 (3 k)     |
| **CompNB**     | 0.7645 / 0.4426     | **0.7707 / 0.4820** | 0.7649 / 0.4640     | 0.4820 (5 k)     | 0.7707 (5 k)     |
| **DecTree**    | **0.7061 / 0.4641** | 0.6995 / 0.4480     | 0.6963 / 0.4503     | 0.4641 (3 k)     | 0.7061 (3 k)     |
| **RandForest** | **0.7760 / 0.4812** | 0.7663 / 0.4587     | 0.7431 / 0.4068     | 0.4812 (3 k)     | 0.7760 (3 k)     |
| **LightGBM**   | 0.4524 / 0.0329     | 0.3246 / 0.0109     | 0.1923 / 0.0142     | 0.0329 (3 k)     | 0.4524 (3 k)     |
| **LinSVC**     | 0.8290 / 0.6678     | 0.8290 / 0.6814     | **0.8295 / 0.6887** | **0.6887 (All)** | 0.8295 (All)     |


- 8개 모델에 대해서 Vocab size는 3000, 5000이 적합해 보임

- 4개 모델에 대해서 비교/분석

| 모델          | 3 000            | 5 000            | 전체                | **추이 요약**    |
| ----------- | ---------------- | ---------------- | ----------------- | ------------ |
| **LogReg**  | 0.799 / **0.48** | 0.798 / **0.48** | 0.792 / **0.45**  | 성능 정체·소폭 하락  |
| **CompNB**  | 0.765 / 0.44     | 0.771 / **0.48** | 0.765 / 0.46      | 5 000에서만 소폭 ↑ |
| **RandFor** | 0.776 / 0.48     | 0.766 / 0.46     | 0.743 / 0.41      | vocab↑→꾸준히 ↓ |
| **LinSVC**  | 0.829 / 0.668    | 0.829 / 0.681    | 0.830 / **0.689** | 안정적·미세 ↑     |


- Logistic -> Vocab size 5000일 때 가장 좋음
- CompNB -> 5000일 때 가장 좋음
- RF -> 3000일 때 가장 좋음
- SVM -> 제한이 없을 때 가장 좋음

> 평균적으로 Vocab size는 5000이 가장 좋아보임

## Vocab size에 따른 모델의 성능 변화의 원인 분석

### LogisticRegression
- L2 정규화가 기본이므로 vocab size가 늘어 고차원의 희소한 피처가 들어와도 큰 차이 없음
- 전체 사전을 사용하면, 희귀어가 늘어 데이터에 비해서 피처의 비율이 증가한다. 따라서 과적합 억제 과정에서 소수 클래스가 피해를 본다.(F1 감소)

### ComplementNB
- NB 모델은 단어의 조건부 확률에 직접적으로 의존한다.
- 일정 수준(5000)까지는 성능이 향상되다가 전체 단어(None)에서 로그 확률이 0에 가까워져 Acc, F1 모두 하락한다.

### RandomForestClassifier
- 트리는 고차원이고 희소한 데이터를 잘 다루지 못한다.
- size가 늘어날수록 성능 하락

### LinearSVC
- 마진 최대 + L2 정규화로 고차원에서도 과적합을 억제
- 새로운 단어가 소수 클래스의 경계를 세밀하게 정의해주는 역할을 해 클래스 불균형에 좋다.(Acc, F1 모두 큰 차이로 좋음)

## 1-D CNN 모델과 SVM 비교

In [31]:
VOCAB = 5000
MAXLEN = 300

(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=VOCAB, test_split=0.2)

print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)

(8982,) (2246,)
(8982,) (2246,)


In [32]:
x_train_seq = pad_sequences(x_train, maxlen=MAXLEN, padding='post')
x_test_seq  = pad_sequences(x_test,  maxlen=MAXLEN, padding='post')

print(x_train_seq.shape, x_test_seq.shape)

(8982, 300) (2246, 300)


In [41]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np


EMB_DIM  = 128
FILTERS  = 128
KERNEL   = 5
DROPOUT  = 0.5
NUM_CLASSES = 46      # Reuters 레이블 수

inp = Input(shape=(MAXLEN,))
x = Embedding(input_dim=VOCAB, output_dim=EMB_DIM)(inp)
x = Conv1D(FILTERS, KERNEL, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dropout(DROPOUT)(x)
x = Dense(64, activation='relu')(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)

cnn = Model(inp, out)
cnn.compile(loss='sparse_categorical_crossentropy',
            optimizer='adam',
            metrics=['accuracy'])

In [42]:
es = EarlyStopping(patience=3, restore_best_weights=True)
history = cnn.fit(x_train_seq, y_train,
                  validation_split=0.1,
                  epochs=20, batch_size=128,
                  callbacks=[es], verbose=2)

Epoch 1/20
64/64 - 2s - loss: 2.7276 - accuracy: 0.3547 - val_loss: 2.0170 - val_accuracy: 0.4950 - 2s/epoch - 31ms/step
Epoch 2/20
64/64 - 1s - loss: 1.8197 - accuracy: 0.5392 - val_loss: 1.6408 - val_accuracy: 0.6274 - 1s/epoch - 16ms/step
Epoch 3/20
64/64 - 2s - loss: 1.4826 - accuracy: 0.6561 - val_loss: 1.3849 - val_accuracy: 0.6930 - 2s/epoch - 25ms/step
Epoch 4/20
64/64 - 2s - loss: 1.2709 - accuracy: 0.6976 - val_loss: 1.2602 - val_accuracy: 0.7152 - 2s/epoch - 25ms/step
Epoch 5/20
64/64 - 2s - loss: 1.1006 - accuracy: 0.7386 - val_loss: 1.1898 - val_accuracy: 0.7286 - 2s/epoch - 26ms/step
Epoch 6/20
64/64 - 2s - loss: 0.9895 - accuracy: 0.7586 - val_loss: 1.1364 - val_accuracy: 0.7430 - 2s/epoch - 25ms/step
Epoch 7/20
64/64 - 2s - loss: 0.8810 - accuracy: 0.7818 - val_loss: 1.0909 - val_accuracy: 0.7542 - 2s/epoch - 24ms/step
Epoch 8/20
64/64 - 2s - loss: 0.7859 - accuracy: 0.8047 - val_loss: 1.0764 - val_accuracy: 0.7586 - 2s/epoch - 24ms/step
Epoch 9/20
64/64 - 2s - loss: 0.

In [55]:
y_prob = cnn.predict(x_test_seq, verbose=0)   # shape: (n_samples, 46)
y_pred = np.argmax(y_prob, axis=1)            # 각 행에서 최대값 인덱스 = 예측 클래스

f1 = f1_score(y_test, y_pred, average="macro")
acc = accuracy_score(y_test, y_pred)
print(f"CNN Acc:{acc:.4f}, F1: {f1:.4f}")

CNN Acc:0.7747, F1: 0.3739


| 모델                      | 입력표현              | Accuracy  | Macro F1   | 비고   |
| ----------------------- | ----------------- |-----------|------------| ---- |
| **Linear SVM** (TF-IDF) | 5 000단어 TF-IDF    | **0.829** | **0.681**  | 실험 값 |
| **1-D CNN**             | 5 000단어 시퀀스 + 임베딩 | **0.775** | 0.374      | 실험 값 |


- Accuracy, F1 모두 SVM이 높다.
1. 훈련 샘플이 8900개 정도에 불과하다. 딥러닝 모델에서는 word embedding에만 128 * 5000 = 640k개의 파라미터가 존재한다. 이렇게 많은 파라미터를 충분히 학습하기에 데이터가 부족하다.
2. 단일 convolution 층으로 모델을 구성했다. 다중 커널(3, 5, 7)이나 여러 layer를 쌓으면 더 좋아질 가능성이 있다. 하지만 파라미터 또한 증가하기 때문에 데이터의 부족이라는 한계를 해결하지는 못할 것으로 예상된다.
3. 사전학습된 Word Embedding을 사용하면 크게 개선될 것으로 생각한다.